In [1]:
pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install tensorflow opencv-python mediapipe scikit-learn matplotlib


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install opencv-python numpy matplotlib mediapipe tensorflow

In [4]:
import cv2 #comp vision library
import numpy as np #lineal algebra
import os #files
from matplotlib import pyplot as plt #image visualization
import time #sleeps
import mediapipe as mp #ML for video

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION) #el 0 -> significa q es una cámara web, si no funciona prueben con 1 o 2
while cap.isOpened ():

  #read feed
  ret, frame = cap.read() #leyendo el frame de la webcam en este momento del tiempo
  #el frame se llamará OpenCV Feed, lo muestra al usuario
  cv2.imshow('OpenCV Feed', frame) #el frame es la imagen de la webcam
  #romperlo
  if cv2.waitKey(10) & 0xFF == ord('q'): #al apretar q se rompe la captura
    break
cap.release()
cv2.destroyAllWindows()

# Puntos clave usando MP Holistic

In [ ]:

mp_holistic = mp.solutions.holistic #Holistic model
mp_drawing = mp.solutions.drawing_utils #Drawing utilities

In [ ]:
def mediapipe_detection(image, model): #esta función recibe una imagen y un modelo holístico 
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) #Conversión de color BGR a RGB blugreenred 2 redgreenblue
    image.flags.writeable = False                  #Ya no se puede modificar la imagen para evitar errores o interferencias
    results = model.process(image)                 #Predicciones para detectar las manos o cara
    image.flags.writeable = True                   #La imagen se puede modificar agan
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) #Se regresa a BGR
    return image, results #imagen y resultados

In [ ]:

def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACE_CONNECTIONS) # Draw face connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

In [ ]:

def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACE_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), 
                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                             ) 
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [ ]:

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()
        if not ret: break

        # Make detections
        image, results = mediapipe_detection(frame, holistic)

        # DEBUG: Si esto imprime, MediaPipe está vivo
        if results.pose_landmarks or results.face_landmarks:
            print("¡Detección activa!") #
            
        print(results)
        
        # Draw landmarks
        draw_landmarks(image, results)

        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release() #
    cv2.destroyAllWindows() #
    for i in range(1,5): cv2.waitKey(1) # Limpieza para macOS

In [ ]:
import cv2
import mediapipe as mp

# Inicializar utilidades
mp_holistic = mp.solutions.holistic 
mp_drawing = mp.solutions.drawing_utils 

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Conversión vital para MediaPipe
    image.flags.writeable = False                  # Mejora rendimiento en M4
    results = model.process(image)                 # PROCESAMIENTO
    image.flags.writeable = True                   
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # Volver a BGR para OpenCV
    return image, results

def draw_landmarks(image, results):
    # Solo dibuja si el objeto no es None
    if results.face_landmarks:
        mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)